In [ ]:
import json
import re
import threading
from groq import Groq
from config import *
from utils import parse_json_response, chunk_narrative
from data_loader import load_writing_standards
import language_tool_python

In [ ]:
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
_tool = None
_tool_thread = None
_tool_lock = threading.Lock()

def get_tool():
    global _tool

    if _tool is None:
        with _tool_lock:
            if _tool is None:
                _tool = language_tool_python.LanguageTool('en-US')

    return _tool


def preload_language_tool():
    global _tool_thread

    if _tool is not None:
        return

    with _tool_lock:
        if _tool is not None:
            return

        if _tool_thread is not None and _tool_thread.is_alive():
            return

        _tool_thread = threading.Thread(
            target=get_tool,
            daemon=True
        )
        _tool_thread.start()


def update_progress(progress_callback, value, desc):
    if progress_callback:
        progress_callback(value, desc=desc)

In [ ]:
def normalize_word(value):
    return re.sub(r"[^A-Za-z0-9'-]", "", str(value)).lower()

def collect_scenario_ignore_words(scenario):
    scenario_text = json.dumps(scenario, ensure_ascii=False)

    # Capture likely generated proper nouns such as names, streets, cities,
    # fictional businesses, hospitals, apartment complexes, and agencies.
    proper_nouns = re.findall(r"\b[A-Z][A-Za-z0-9'’-]*\b", scenario_text)

    ignore_words = {
        normalize_word(word)
        for word in proper_nouns
        if len(normalize_word(word)) > 1
    }

    # Always include interviewee name parts, even if formatting changes.
    for word in str(scenario.get("interviewee_name", "")).split():
        normalized = normalize_word(word)
        if normalized:
            ignore_words.add(normalized)

    return ignore_words


def should_skip_language_tool_match(match, chunk_text, scenario_ignore_words):
    spelling_rule_ids = {
        "MORFOLOGIK_RULE_EN_US"
    }

    if match.rule_id not in spelling_rule_ids:
        return False

    flagged_text = chunk_text[match.offset:match.offset + match.error_length]
    flagged_words = [
        normalize_word(word)
        for word in re.findall(r"\b[A-Za-z0-9'’-]+\b", flagged_text)
    ]
    flagged_words = [word for word in flagged_words if word]

    if not flagged_words:
        return False

    return all(word in scenario_ignore_words for word in flagged_words)

In [ ]:
def check_narrative_coverage(chunks, checklist, progress_callback=None, progress_start=0.15, progress_end=0.35):
    elements = checklist["required_elements"]

    elements_text = "\n".join([
        f'  {el["id"]}: {el["element"]}'
        for el in elements
    ])

    coverage = {
        el["id"]: {
            "covered": False,
            "phrases": {}
        }
        for el in elements
    }

    total_chunks = max(len(chunks), 1)

    for chunk_number, (chunk_index, chunk_text) in enumerate(chunks.items(), start=1):
        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * (chunk_number - 1) / total_chunks),
            f"Checking narrative coverage, chunk {chunk_number} of {total_chunks}..."
        )

        prompt = f"""You are analyzing a section of a police report narrative to determine which required reporting elements it addresses.

Crime type: {checklist["crime_type"]}

COVERAGE RUBRIC:
- "full": the narrative text directly and completely addresses the element
- "partial": the narrative text touches on the element but is incomplete or vague
- none: the narrative text has no meaningful connection to the element — do not include in output

EXAMPLES:
Element: "Emotional state of the victim"
- Full: "The victim was visibly shaking and crying throughout the interview. She stated she was terrified he would return."
- Partial: "The victim appeared upset."
- None: "The suspect left the scene prior to officer arrival."

Element: "History of the relationship / previous incidents"
- Full: "The victim stated she and the suspect have been married for six years and that he has assaulted her on approximately ten prior occasions over the past two years."
- Partial: "The victim stated this has happened before."
- None: "Officers photographed the scene."

Element: "Weapons or objects used"
- Full: "The victim stated the suspect struck her with a closed fist and then grabbed a kitchen knife and held it to her throat."
- Partial: "The victim stated the suspect had a weapon."
- None: "The victim requested medical attention."

Element: "All injuries visible and non-visible documented"
- Full: "The victim had a visible bruise on her left cheekbone and complained of pain in her ribs. She also reported pain when swallowing consistent with possible strangulation."
- Partial: "The victim had a bruise on her face."
- None: "The suspect was not present upon officer arrival."

Element: "All threats clearly documented"
- Full: "The victim stated the suspect said 'I will kill you if you call the police again' during the incident."
- Partial: "The victim stated the suspect threatened her."
- None: "The victim signed a medical release form."

Element: "Medical attention"
- Full: "The victim was transported by ambulance to St. Mary's Hospital. She refused further treatment and signed a medical release form."
- Partial: "The victim was offered medical attention."
- None: "The suspect has a prior domestic violence conviction."

Here are all the required checklist elements:
{elements_text}

Here is the narrative section to analyze (chunk {chunk_index}):
{chunk_text}

For each element that this narrative section addresses (fully or partially), return a JSON object.
Only include elements that are addressed in this specific section.

Important matching rules:
- The same phrase may support more than one checklist element. If one phrase clearly supports multiple elements, include all of those elements.
- Do not choose only the closest or most specific checklist element if the phrase also supports another required element.
- Relationship facts may support both relationship-identification elements and relationship-history elements when appropriate.
- Threat statements may support both threat-documentation elements and fear/safety elements when appropriate.
- Concrete negative facts count as coverage when the element asks whether something occurred. For example, "no injuries were reported" may cover an injury-documentation element, and "no weapons were mentioned" may cover a weapons element.
- Do not count headings, section titles, checklist-question text, or empty strings as evidence.
- The phrase must be an exact quote from the narrative section and must contain factual content.

If no elements are addressed, return an empty JSON object {{}}.

Return ONLY a valid JSON object, no markdown fences:
{{
  "element_id": {{
    "coverage": "full" or "partial",
    "phrase": "exact quote from the narrative that addresses this element"
  }},
  ...
}}"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP_DETERMINISTIC,
            response_format={"type": "json_object"}
        )

        result = parse_json_response(response.choices[0].message.content)

        for element_id, data in result.items():
            if element_id not in coverage:
                continue

            current = coverage[element_id]["covered"]
            new_coverage = data["coverage"]
            phrase = data.get("phrase", "")

            # Always record the phrase
            coverage[element_id]["phrases"][chunk_index] = phrase

            # Update coverage status
            if current == False:
                coverage[element_id]["covered"] = new_coverage

            elif current == "partial":
                if new_coverage == "full":
                    coverage[element_id]["covered"] = "full"
                else:
                    # Re-evaluate combined phrases
                    combined_phrases = "\n".join([
                        f'  Chunk {idx}: {p}'
                        for idx, p in coverage[element_id]["phrases"].items()
                    ])

                    element_text = next(
                        el["element"] for el in elements
                        if el["id"] == element_id
                    )

                    reeval_prompt = f"""A police report narrative addresses a required reporting element across multiple sections.

Required element: {element_text}

Narrative phrases so far:
{combined_phrases}

Do these phrases together constitute full coverage of this element, or is it still only partial?
Return ONLY a valid JSON object:
{{"coverage": "full" or "partial"}}"""

                    reeval_response = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{"role": "user", "content": reeval_prompt}],
                        temperature=TEMP_DETERMINISTIC,
                        response_format={"type": "json_object"}
                    )

                    reeval = parse_json_response(reeval_response.choices[0].message.content)
                    coverage[element_id]["covered"] = reeval["coverage"]

            # If already "full", just record the phrase — don't downgrade

        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * chunk_number / total_chunks),
            f"Finished narrative coverage chunk {chunk_number} of {total_chunks}."
        )

    return coverage

In [ ]:
def grade_content(narrative_coverage, scenario, checklist, coverage_flags, progress_callback=None, progress_start=0.35, progress_end=0.50):
    results = []
    elements = checklist["required_elements"]
    total_elements = max(len(elements), 1)

    for element_number, el in enumerate(elements, start=1):
        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * (element_number - 1) / total_elements),
            f"Validating checklist element {element_number} of {total_elements}..."
        )

        element_id = el["id"]
        element_text = el["element"]

        interview_data = coverage_flags.get(element_id, {
            "covered": False,
            "messages": {}})
        narrative_data = narrative_coverage.get(element_id, {
            "covered": False,
            "phrases": {}})
        response_data = {
            "covered": interview_data.get("response_covered", False),
            "messages": interview_data.get("response_messages", {})
        }

        # Validate interview coverage flag if it claims coverage
        validated_interview_coverage = interview_data["covered"]
        if interview_data["covered"] and interview_data["messages"]:
            combined_messages = "\n".join([
                f'  Line {line}: {msg["officer_message"]}'
                for line, msg in interview_data["messages"].items()
            ])

            validation_prompt = f"""You are validating whether a police officer's interview questions actually address a required reporting element.

Required element: {element_text}

Officer questions recorded as covering this element:
{combined_messages}

Do these questions genuinely address this element?
- "full": the questions would clearly elicit a complete answer for this element
- "partial": the questions touch on this element but would only elicit an incomplete answer
- false: the questions do not meaningfully address this element

Return ONLY a valid JSON object:
{{"coverage": "full" or "partial" or false}}"""

            validation_response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": validation_prompt}],
                temperature=TEMP_DETERMINISTIC,
                response_format={"type": "json_object"}
            )

            reeval = parse_json_response(validation_response.choices[0].message.content)
            validated_interview_coverage = reeval["coverage"]

        # Validate interviewee response coverage flag if it claims coverage
        validated_response_coverage = response_data["covered"]
        if response_data["covered"] and response_data["messages"]:
            combined_responses = "\n".join([
                f'  Line {line}:\n    Officer: {msg["officer_message"]}\n    Interviewee: {msg["interviewee_response"]}\n    Provided fact: {msg.get("provided_fact", "")}'
                for line, msg in response_data["messages"].items()
            ])

            response_validation_prompt = f"""You are validating whether an interviewee's answers actually provided information for a required reporting element.

Required element: {element_text}

Interviewee answers recorded as providing this element:
{combined_responses}

Do these answers genuinely provide information for this element?
- "full": the interviewee provided enough information to document this element completely
- "partial": the interviewee provided some information about this element but not enough to document it completely
- false: the interviewee did not meaningfully provide information for this element

Return ONLY a valid JSON object:
{{"coverage": "full" or "partial" or false}}"""

            response_validation_response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": response_validation_prompt}],
                temperature=TEMP_DETERMINISTIC,
                response_format={"type": "json_object"}
            )

            response_reeval = parse_json_response(response_validation_response.choices[0].message.content)
            validated_response_coverage = response_reeval["coverage"]

        if validated_interview_coverage in ("full", "partial"):
            final_interview_coverage = validated_interview_coverage
            final_interview_messages = interview_data["messages"]
            interview_source = "officer_question"
            interview_source_label = "asked by officer"

        elif validated_response_coverage in ("full", "partial"):
            final_interview_coverage = validated_response_coverage
            final_interview_messages = response_data["messages"]
            interview_source = "interviewee_response"
            interview_source_label = f'provided by {scenario.get("interviewee_role", "interviewee")}'

        else:
            final_interview_coverage = False
            final_interview_messages = {}
            interview_source = None
            interview_source_label = None

        # Build notes using final interview coverage
        notes = []

        if final_interview_coverage and not narrative_data["covered"]:
            notes.append("Asked or provided in interview but not written up in narrative.")
        if narrative_data["covered"] and not final_interview_coverage:
            notes.append("Appears in narrative but was not asked by the officer or provided by the interviewee.")
        if validated_interview_coverage != interview_data["covered"]:
            notes.append("Interview coverage flag could not be validated against recorded questions.")
        if validated_response_coverage != response_data["covered"]:
            notes.append("Interviewee-provided coverage flag could not be validated against recorded responses.")
        if interview_source == "interviewee_response":
            notes.append("Information was provided by the interviewee during the interview.")
        if narrative_data["covered"] == "partial":
            notes.append("Narrative addresses this element but only partially.")
        if final_interview_coverage == "partial" and narrative_data["covered"] == "partial":
            notes.append("Both interview and narrative only partially cover this element.")

        results.append({
            "element_id": element_id,
            "element": element_text,
            "asked_in_interview": final_interview_coverage,
            "interview_messages": final_interview_messages if final_interview_coverage in ("full", "partial") else {},
            "interview_source": interview_source,
            "interview_source_label": interview_source_label,
            "in_narrative": narrative_data["covered"],
            "narrative_phrases": narrative_data["phrases"] if narrative_data["covered"] in ("full", "partial") else {},
            "notes": " ".join(notes) if notes else None
        })

        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * element_number / total_elements),
            f"Finished checklist element {element_number} of {total_elements}."
        )

    return results

In [ ]:
def detect_contradictions(coverage_dict, scenario, checklist, source, progress_callback=None, progress_start=0.50, progress_end=0.65):
    contradictions = []
    items = list(coverage_dict.items())
    total_items = max(len(items), 1)

    for item_number, (element_id, data) in enumerate(items, start=1):
        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * (item_number - 1) / total_items),
            f"Checking {source} contradictions, item {item_number} of {total_items}..."
        )

        if not data["covered"]:
            update_progress(
                progress_callback,
                progress_start + ((progress_end - progress_start) * item_number / total_items),
                f"Checked {source} contradictions, item {item_number} of {total_items}."
            )
            continue

        scenario_fact = scenario["facts"].get(element_id)
        if not scenario_fact:
            update_progress(
                progress_callback,
                progress_start + ((progress_end - progress_start) * item_number / total_items),
                f"Checked {source} contradictions, item {item_number} of {total_items}."
            )
            continue

        element_text = next(
            (el["element"] for el in checklist["required_elements"] if el["id"] == element_id),
            None
        )

        if source == "interview":
            texts = {
                line: msg["officer_message"]
                for line, msg in data["messages"].items()
            }
        else:
            texts = data["phrases"]

        for location, text in texts.items():
            prompt = f"""You are checking whether a piece of text contradicts an established scenario fact.

Established scenario fact for element "{element_text}":
{scenario_fact}

Text to check ({source}, location {location}):
{text}

Does this text directly contradict the scenario fact?
Only flag a genuine contradiction — where the text states something that directly conflicts with the fact.
Do not flag missing information, vague language, or partial coverage — only direct conflicts.

Return ONLY a valid JSON object:
{{
  "contradiction": true or false,
  "phrase": "exact quote from the text that contradicts the fact, or null",
  "explanation": "why this is a contradiction, or null"
}}"""

            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMP_DETERMINISTIC,
                response_format={"type": "json_object"}
            )

            result = parse_json_response(response.choices[0].message.content)

            if result["contradiction"]:
                contradictions.append({
                    "element_id": element_id,
                    "element": element_text,
                    "location": location,
                    "source": source,
                    "phrase": result["phrase"],
                    "contradicts": scenario_fact,
                    "explanation": result["explanation"]
                })

        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * item_number / total_items),
            f"Checked {source} contradictions, item {item_number} of {total_items}."
        )

    return contradictions

In [ ]:
def summarize_chunks(chunks, progress_callback=None, progress_start=0.02, progress_end=0.15):
    summaries = {}
    total_chunks = max(len(chunks), 1)

    for chunk_number, (chunk_index, chunk_text) in enumerate(chunks.items(), start=1):
        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * (chunk_number - 1) / total_chunks),
            f"Summarizing report chunk {chunk_number} of {total_chunks}..."
        )

        prompt = f"""Summarize the following section of a police report narrative in exactly 2 sentences.
Focus on the sequence of events and key facts described.

Section:
{chunk_text}

Return only the 2 sentence summary, no additional text."""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP_DETERMINISTIC
        )

        summaries[chunk_index] = response.choices[0].message.content.strip()

        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * chunk_number / total_chunks),
            f"Finished summary chunk {chunk_number} of {total_chunks}."
        )

    return summaries

In [ ]:
def grade_grammar(chunks, scenario_ignore_words=None, progress_callback=None, progress_start=0.66, progress_end=0.80):
    errors = []
    scenario_ignore_words = scenario_ignore_words or set()
    total_chunks = max(len(chunks), 1)

    for chunk_number, (chunk_index, chunk_text) in enumerate(chunks.items(), start=1):
        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * (chunk_number - 1) / total_chunks),
            f"Checking grammar, chunk {chunk_number} of {total_chunks}..."
        )

        # LanguageTool pass
        lt_matches = get_tool().check(chunk_text)
        for match in lt_matches:
            if should_skip_language_tool_match(match, chunk_text, scenario_ignore_words):
                continue

            errors.append({
                "chunk": chunk_index,
                "source": "language_tool",
                "error_type": match.rule_id,
                "excerpt": chunk_text[match.offset:match.offset + match.error_length],
                "suggestion": match.replacements[0] if match.replacements else None
            })

        # LLM pass
        prompt = f"""You are evaluating a section of a police report for grammar errors that require understanding context across sentences.

Check ONLY for:
- Pronoun clarity issues where it is ambiguous who a pronoun refers to
- Verb tense inconsistencies across sentences
- Sentence fragments that span multiple clauses
- Run-on sentences

Do NOT flag spelling, punctuation, or single-word errors — those are handled separately.

Section (chunk {chunk_index}):
{chunk_text}

For each error found return an entry in the JSON array.
If no errors are found return an empty array.

Return ONLY a valid JSON array, no markdown fences:
[
  {{
    "chunk": {chunk_index},
    "source": "llm",
    "error_type": "type of grammar error",
    "excerpt": "exact quote from the text containing the error",
    "suggestion": "corrected version"
  }},
  ...
]"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP_DETERMINISTIC
        )

        result = parse_json_response(response.choices[0].message.content)
        errors.extend(result)

        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * chunk_number / total_chunks),
            f"Finished grammar chunk {chunk_number} of {total_chunks}."
        )

    return errors

In [ ]:
def grade_style(chunks, summaries, progress_callback=None, progress_start=0.80, progress_end=0.90):
    writing_standards = load_writing_standards()
    errors = []

    summaries_text = "\n".join([
        f"  Chunk {idx}: {summary}"
        for idx, summary in summaries.items()
    ])

    total_chunks = max(len(chunks), 1)

    for chunk_number, (chunk_index, chunk_text) in enumerate(chunks.items(), start=1):
        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * (chunk_number - 1) / total_chunks),
            f"Checking style, chunk {chunk_number} of {total_chunks}..."
        )

        prompt = f"""You are evaluating a section of a police report for professional writing style errors specific to law enforcement report writing.

WRITING STANDARDS — evaluate against these rules:
{writing_standards}

NARRATIVE STRUCTURE OVERVIEW — summaries of each section in order, for evaluating chronology and document-level structure:
{summaries_text}

Section to evaluate (chunk {chunk_index}):
{chunk_text}

Check for violations of the writing standards above, including both sentence-level issues and any structural issues visible from the summaries.

For each violation found return an entry in the JSON array.
If no violations are found return an empty array.

Return ONLY a valid JSON array, no markdown fences:
[
  {{
    "chunk": {chunk_index},
    "rule": "which writing standard was violated, citing the rule number",
    "error_type": "passive voice / conclusory statement / future tense / jargon / chronological order / etc.",
    "excerpt": "exact quote from the chunk containing the violation",
    "suggestion": "corrected version"
  }},
  ...
]"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP_DETERMINISTIC,
            response_format={"type": "json_object"}
        )

        result = parse_json_response(response.choices[0].message.content)
        errors.extend(result.get("errors", []))

        update_progress(
            progress_callback,
            progress_start + ((progress_end - progress_start) * chunk_number / total_chunks),
            f"Finished style chunk {chunk_number} of {total_chunks}."
        )

    return errors

In [ ]:
def grade_report(narrative, scenario, checklist, coverage_flags, progress_callback=None):
    update_progress(progress_callback, 0.05, "Splitting report into chunks...")
    chunks = chunk_narrative(narrative)

    update_progress(progress_callback, 0.10, "Summarizing report chunks...")
    summaries = summarize_chunks(
        chunks,
        progress_callback=progress_callback,
        progress_start=0.10,
        progress_end=0.20
    )

    update_progress(progress_callback, 0.20, "Checking narrative coverage...")
    narrative_coverage = check_narrative_coverage(
        chunks,
        checklist,
        progress_callback=progress_callback,
        progress_start=0.20,
        progress_end=0.40
    )

    update_progress(progress_callback, 0.40, "Validating checklist coverage...")
    content = grade_content(
        narrative_coverage,
        scenario,
        checklist,
        coverage_flags,
        progress_callback=progress_callback,
        progress_start=0.40,
        progress_end=0.58
    )

    update_progress(progress_callback, 0.58, "Checking contradictions...")
    contradictions_interview = detect_contradictions(
        coverage_flags,
        scenario,
        checklist,
        "interview",
        progress_callback=progress_callback,
        progress_start=0.58,
        progress_end=0.64
    )

    contradictions_narrative = detect_contradictions(
        narrative_coverage,
        scenario,
        checklist,
        "narrative",
        progress_callback=progress_callback,
        progress_start=0.64,
        progress_end=0.70
    )

    update_progress(progress_callback, 0.70, "Checking grammar...")
    grammar = grade_grammar(
        chunks,
        scenario_ignore_words=collect_scenario_ignore_words(scenario),
        progress_callback=progress_callback,
        progress_start=0.70,
        progress_end=0.82
    )

    update_progress(progress_callback, 0.82, "Checking style...")
    style = grade_style(
        chunks,
        summaries,
        progress_callback=progress_callback,
        progress_start=0.82,
        progress_end=0.90
    )

    return {
        "content": content,
        "contradictions_interview": contradictions_interview,
        "contradictions_narrative": contradictions_narrative,
        "grammar": grammar,
        "style": style,
        "narrative_coverage": narrative_coverage  # added
    }